In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
import pandas as pd
import time

# ---------- INPUTS ----------
role_input = input("Enter job role (e.g., data-analyst): ").strip().replace(" ", "-")
exp_input = input("Enter experience (in years, e.g., 0, 2, 5): ").strip()
location_input = input("Enter location (e.g., hyderabad, bangalore) or leave blank for all: ").strip().replace(" ", "-")

# ---------- SETUP ----------
driver = webdriver.Chrome()
jobs = {
    # "job_no": [],
    "roles": [],
    "companies": [],
    "locations": [],
    "experience": [],
    "skills": [],
    "links": [],
    # "description": []
}

# ---------- SCRAPING ----------
for i in range(1, 8):  # scrape first 7 pages
    if location_input:
        url = f"https://www.naukri.com/{role_input}-jobs-in-{location_input}-{i}?experience={exp_input}"
    else:
        url = f"https://www.naukri.com/{role_input}-jobs-{i}?experience={exp_input}"

    print(f"Scraping: {url}")
    driver.get(url)
    time.sleep(3)

    lst = driver.find_elements(By.CLASS_NAME, "srp-jobtuple-wrapper")

    for index, job in enumerate(lst):
        driver.implicitly_wait(10)
        jobno = (i - 1) * len(lst) + index + 1

        def safe_find(class_name):
            try:
                return job.find_element(By.CLASS_NAME, class_name).text
            except NoSuchElementException:
                return "NA"

        role = safe_find("title")
        company = safe_find("comp-name")
        location = safe_find("loc-wrap")
        exp = safe_find("exp-wrap")
        # desc = safe_find("job-desc")

        # --- Extract job link ---
        try:
            link = job.find_element(By.CLASS_NAME, "title").get_attribute("href")
        except NoSuchElementException:
            link = "NA"

        # --- Extract skills ---
        try:
            skill_ul = job.find_element(By.CLASS_NAME, "tags-gt")
            skill_li = skill_ul.find_elements(By.TAG_NAME, "li")
            skill_tag = [li.text for li in skill_li]
            skill = ', '.join(skill_tag)
        except NoSuchElementException:
            skill = "NA"

        # --- Append to dictionary ---
        # jobs["job_no"].append(jobno)
        jobs["roles"].append(role)
        jobs["companies"].append(company)
        jobs["locations"].append(location)
        jobs["experience"].append(exp)
        jobs["skills"].append(skill)
        jobs["links"].append(link)
        # jobs["description"].append(desc)

# ---------- SAVE TO CSV ----------
df = pd.DataFrame(jobs)
csv_filename = f"csv/naukri_jobs.csv"
df.to_csv(csv_filename, index=False, encoding="utf-8-sig")

print(f"\n✅ Data saved successfully as '{csv_filename}'")

driver.quit()

Scraping: https://www.naukri.com/Machine-Learning-Engineer-jobs-in-Hyderabad-1?experience=0
Scraping: https://www.naukri.com/Machine-Learning-Engineer-jobs-in-Hyderabad-2?experience=0
Scraping: https://www.naukri.com/Machine-Learning-Engineer-jobs-in-Hyderabad-3?experience=0
Scraping: https://www.naukri.com/Machine-Learning-Engineer-jobs-in-Hyderabad-4?experience=0
Scraping: https://www.naukri.com/Machine-Learning-Engineer-jobs-in-Hyderabad-5?experience=0
Scraping: https://www.naukri.com/Machine-Learning-Engineer-jobs-in-Hyderabad-6?experience=0
Scraping: https://www.naukri.com/Machine-Learning-Engineer-jobs-in-Hyderabad-7?experience=0

✅ Data saved successfully as 'csv/naukri_jobs.csv'
